In [29]:

import pandas as pd
from cmdstanpy import CmdStanModel

In [ ]:
data = pd.read_csv("data/response_times.csv", sep=";")

J = int(data["id"].nunique())

data_dict = {
    "N": int(len(data)),
    "J": J,
    "id": data["id"].astype(int).tolist(),
    "y": data["rt"].astype(float).tolist(),
    "choice": data["choice"].astype(int).tolist(),
    "condition": data["condition"].astype(int).tolist(),
}

data.head()


,id,rt,choice,condition
0,1,1.4456,1,1
1,1,1.7426,1,1
2,1,2.9506,1,1
3,1,1.0676,1,1
4,1,4.9116,1,1


In [31]:


model = CmdStanModel(stan_file="models/diffusion_multiple.stan")


In [32]:
fit = model.sample(
    data=data_dict,
    chains=4,
    parallel_chains=4,
    iter_warmup=500,
    iter_sampling=5000,
    seed=42,
    inits=0.2,
    show_progress=False,
)

summary = fit.summary()


20:20:02 - cmdstanpy - INFO - CmdStan start processing
20:20:02 - cmdstanpy - INFO - Chain [1] start processing
20:20:02 - cmdstanpy - INFO - Chain [2] start processing
20:20:02 - cmdstanpy - INFO - Chain [3] start processing


20:20:02 - cmdstanpy - INFO - Chain [4] start processing
20:20:40 - cmdstanpy - INFO - Chain [2] done processing
20:20:40 - cmdstanpy - INFO - Chain [4] done processing
20:20:40 - cmdstanpy - INFO - Chain [3] done processing
20:20:42 - cmdstanpy - INFO - Chain [1] done processing
20:20:42 - cmdstanpy - WARNING - Non-fatal error during sampling:
Exception: wiener_lpdf: Boundary separation is inf, but must be positive finite! (in 'diffusion_multiple.stan', line 61, column 16 to line 66, column 18)
	Exception: wiener_lpdf: Boundary separation is inf, but must be positive finite! (in 'diffusion_multiple.stan', line 61, column 16 to line 66, column 18)
Exception: wiener_lpdf: Boundary separation is 0, but must be positive finite! (in 'diffusion_multiple.stan', line 61, column 16 to line 66, column 18)
Exception: wiener_lpdf: Boundary separation is 0, but must be positive finite! (in 'diffusion_multiple.stan', line 61, column 16 to line 66, column 18)
Exception: wiener_lpdf: Boundary separat

In [33]:
drift_summary = pd.DataFrame({
    "operator": list(range(1, J + 1)),
    "v_condition_1": [summary.loc[f"v1[{j}]", "Mean"] for j in range(1, J + 1)],
    "v_condition_2": [summary.loc[f"v2[{j}]", "Mean"] for j in range(1, J + 1)],
})

drift_summary["harder_condition"] = [
    1 if row.v_condition_1 < row.v_condition_2 else 2
    for row in drift_summary.itertuples()
]

drift_summary.round(3)


,operator,v_condition_1,v_condition_2,harder_condition
0,1,0.829,3.561,1
1,2,0.731,3.823,1
2,3,0.669,3.439,1
3,4,0.478,3.381,1
4,5,0.263,3.454,1


In [34]:
avg_v1 = drift_summary["v_condition_1"].mean()
avg_v2 = drift_summary["v_condition_2"].mean()

print(f"Average drift for condition 1: {avg_v1:.3f}")
print(f"Average drift for condition 2: {avg_v2:.3f}")

if avg_v1 < avg_v2:
    print("Condition 1 is the more difficult condition.")
else:
    print("Condition 2 is the more difficult condition.")


Average drift for condition 1: 0.594
Average drift for condition 2: 3.532
Condition 1 is the more difficult condition.


In [ ]:
print(f"Max R-hat: {summary['R_hat'].max():.4f}")
print(f"Min bulk ESS: {summary['ESS_bulk'].min():.1f}")
print(f"Min tail ESS: {summary['ESS_tail'].min():.1f}")
print(fit.diagnose())


Max R-hat: 1.0006
Min bulk ESS: 8258.9
Min tail ESS: 12242.6

Checking sampler transitions treedepth.
Treedepth satisfactory for all transitions.

Checking sampler transitions for divergences.
No divergent transitions found.

Checking E-BFMI - sampler transitions HMC potential energy.
E-BFMI satisfactory.

Rank-normalized split effective sample size satisfactory for all parameters.

Rank-normalized split R-hat values satisfactory for all parameters.

Processing complete, no problems detected.

